In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
# from helper import load_env
# load_env()
from pydantic import BaseModel, Field
from typing import List, Dict, Type
from typing import List, Optional
import os
import yaml

In [2]:
import os, json, time, gc
import logging 

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, Image, Video
from tqdm import tqdm
from openai import OpenAI, AsyncOpenAI
from openai.types.chat import (ChatCompletion, 
                               ChatCompletionChunk,
                               ChatCompletionContentPartTextParam, 
                               ChatCompletionContentPartImageParam,
                               ChatCompletionStreamOptionsParam)
import asyncio
import aiohttp
import pandas as pd
import re


import base64
from PIL import Image
import io

#fix bug with aysncio and jupyter
import nest_asyncio # for langchain async 
nest_asyncio.apply()

In [3]:
import litellm
from litellm import acompletion, completion

### Test LM Studio Connection By Openai API

In [4]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
VLLM_URL = "http://localhost:8000/v1"
api_key= "lm-studio"

In [12]:
# client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=api_key)
client = OpenAI(base_url=VLLM_URL, api_key=api_key)

# Replace with the exact model name running in LM Studio
model_name = "qwen3.6-35b-a3b-mtp"  #"google/gemma-4-12b" 
model_name = "Qwen/Qwen3-0.6B"

In [13]:
ret = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a helpful AI coding assistant."},
        {"role": "user", "content": "Explain how to check memory usage in a Jupyter notebook."}
    ],
    temperature=0.7,
)

Markdown(ret.choices[0].message.content)

<think>
Okay, the user wants to know how to check memory usage in a Jupyter notebook. Let me start by recalling the different methods available. First, I remember that in Jupyter, there's a `jupyter_nbplot` function which plots a graph, but that's more for visualization. The user probably wants to check memory, so maybe they're using a script or a notebook that's analyzing data.

Wait, the user might be using Jupyter for data analysis, so checking memory usage could involve things like memory allocation or using tools like `memory_usage` or `pandas`. Also, there's the `ipymem` module, which might be relevant. But I need to make sure the steps are clear.

I should explain each method. First, using `pandas`'s `memory_usage` function. That's a standard approach. Then, using `ipymem` which is a module for memory analysis. Also, using `jupyter` commands, like `jupyter nbinfo` which shows the notebook's memory usage. But maybe the user wants a script, so they can run a Python script to analyze memory.

Wait, but the user asked to explain, not code. So I need to present the steps in a way that's easy to follow. Also, mention that different methods can be used depending on the environment. Let me check if there are any other methods, like using `heapq` or `sys` modules, but those might be more advanced. Also, note that memory usage can vary based on the data type and operations.

Make sure to list each method with a brief description. Maybe start with the `pandas` approach, then the `ipymem` module, and finally the Jupyter commands. Also, mention that the results can vary and that some tools are more efficient for certain tasks. That should cover the user's needs.
</think>

To check memory usage in a Jupyter notebook, you can use various methods. Here are the standard approaches:

1. **Using `pandas`'s `memory_usage`**:
   - This function calculates memory usage for a dataset. It returns a dictionary with keys like `'memory_usage'` and the values representing memory usage in bytes.
   - Example:
     ```python
     import pandas as pd
     data = pd.DataFrame({'A': [1, 2, 3]})
     memory_usage = pd.DataFrame(memory_usage=data)
     print(memory_usage)
     ```

2. **Using `ipymem` (a Jupyter module)**:
   - This module provides tools to analyze memory usage in a Jupyter notebook. It can output memory usage in bytes or other formats.
   - Example:
     ```python
     import ipymem
     ipymem.memory_usage()
     ```

3. **Using `jupyter` commands**:
   - You can check memory usage by running the `jupyter nbinfo` command:
     ```bash
     jupyter nbinfo <notebook_id>
     ```
   - This command displays the notebook's memory usage in bytes and other relevant information.

### Notes:
- Memory usage can vary based on the data type, operations, and notebook environment.
- Tools like `ipymem` are especially useful for efficient memory analysis in Jupyter environments.

## Test LM studio LLM Connection by liteLLM API

In [14]:
# Markdown(completion.choices[0].message.content)

In [15]:
# 3. Define the async function
async def get_chat_completion(
    api_base,
    api_key, 
    model_name = "openai/local-model",
    system_prompt= "You are a helpful assistant.",
    user_prompt="",
    temperature= 0.7,
    max_tokens=4096):
    
    response = await acompletion(
        model=model_name,  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
        api_base=api_base,
        api_key=api_key,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response

In [18]:
%%time
# 4. Execute the async function directly in Jupyter
# response = asyncio.run(get_chat_completion(api_base=LM_STUDIO_BASE_URL, 
#                                            api_key=api_key,
#                                            model_name="openai/local-model",
#                                             user_prompt="What is LLM?"))

response = asyncio.run(get_chat_completion(api_base=VLLM_URL, 
                                           api_key=api_key,
                                           model_name="hosted_vllm/" + model_name,
                                            user_prompt="What is LLM?"))



CPU times: user 4.85 ms, sys: 1.89 ms, total: 6.74 ms
Wall time: 836 ms


In [19]:
Markdown(response.choices[0].message.content)

<think>
Okay, the user asked, "What is LLM?" I need to explain what a Large Language Model (LLM) is. Let me start by recalling what I know about LLMs.

First, they're artificial intelligence models. So, LLMs are big neural networks trained on vast amounts of text data. They can understand and generate human language, right? That makes them useful in various applications like chatbots, content generation, and language processing.

Wait, but I should mention that they're trained on a huge dataset, so they can handle different languages and topics. Also, LLMs can perform tasks like summarizing text, answering questions, or even writing creative content. I need to make sure the explanation is clear and covers the key points without getting too technical.

Let me check if there's any confusion. LLMs are not just machines that can speak; they can understand and create language. So, maybe mention that they use deep learning techniques and have large training data. Also, emphasize their applications in real life. I should keep it simple and direct, maybe start with a definition and then list the features and applications.
</think>

A Large Language Model (LLM) is a type of artificial intelligence model developed to understand and generate human language. These models are trained on vast datasets of text, allowing them to comprehend and process complex language patterns, understand context, and generate coherent responses. They can perform tasks like summarizing information, answering questions, writing creative content, and even understand multiple languages. LLMs rely on deep learning techniques and are used in various applications such as chatbots, language translation, and content creation.

## Concurrent Version 

In [20]:
import os
import pandas as pd
import asyncio
from tqdm.asyncio import tqdm_asyncio
from litellm import acompletion
import time

In [21]:
LM_STUDIO_BASE_URL = "http://localhost:1234/v1"
api_key = "lm-studio"
MAX_CONCURRENT = 2
DELAY = 1        # small delay between batches (optional)
BATCH_SIZE = 20 #50      # process in batches for safer saving (number of row)
# set 
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

In [10]:
# ====================== ASYNC GENERATE FUNCTION ======================
async def async_generate_cot_data(prompt: str, answer: str) -> str:
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.
Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}
Correct Answer: {answer}
Please think step by step inside <think> tags about how to discover the transformation rule."""

    async with semaphore:
        try:
            response = await acompletion(
                model="openai/local-model",
                api_base=LM_STUDIO_BASE_URL,
                api_key=api_key,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.3,
                max_tokens=1600,
                timeout=180
            )

            message = response.choices[0].message
            reasoning = getattr(message, "reasoning_content", "") or ""
            content = message.content or ""

            if reasoning:
                thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
            else:
                thinking_part = f"<think>\n{content.strip()}\n</think>"

            return f"{thinking_part}\n\\boxed{{{answer}}}"

        except Exception as e:
            print(f"Error: {e}")
            return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"

In [11]:
async def generate_cot_with_resume():
    # === Resume Logic ===
    '''
    for concurrent version
    '''
    if os.path.exists(outputFile):
        print("Found existing train_cot.csv → Resuming...")
        trainDF = pd.read_csv(outputFile)
    else:
        print("No existing file found. Starting from train.csv...")
        trainDF = pd.read_csv(trainFile)
        if "cot_reasoning" not in trainDF.columns:
            trainDF["cot_reasoning"] = ""

    # Count remaining rows
    remaining_mask = trainDF["cot_reasoning"].isna() | (trainDF["cot_reasoning"] == "")
    remaining = remaining_mask.sum()

    print(f"Total rows: {len(trainDF)}")
    print(f"Rows already processed: {len(trainDF) - remaining}")
    print(f"Rows left to process: {remaining}\n")

    if remaining == 0:
        print("✅ All rows already have CoT reasoning. Nothing to do.")
        return

    # Get indices that need processing
    indices_to_process = trainDF[remaining_mask].index.tolist()
    print(f"Starting CoT generation with {MAX_CONCURRENT} concurrent requests...\n")

    processed_count = 0

    # Process in batches for safer saving
    for start in range(0, len(indices_to_process), BATCH_SIZE):
        batch_indices = indices_to_process[start : start + BATCH_SIZE]
        batch_tasks = []

        for idx in batch_indices:
            prompt = trainDF.loc[idx, "prompt"]
            answer = str(trainDF.loc[idx, "answer"]).strip()
            task = async_generate_cot_data(prompt, answer)
            batch_tasks.append((idx, task))

        # Run batch concurrently
        results = await tqdm_asyncio.gather(
            *[task for _, task in batch_tasks],
            desc=f"Batch {start // BATCH_SIZE + 1}"
        )

        # Update dataframe
        for (idx, _), result in zip(batch_tasks, results):
            trainDF.loc[idx, "cot_reasoning"] = result
            processed_count += 1

        # Save progress after each batch
        trainDF.to_csv(outputFile, index=False)
        print(f"Saved progress. Processed {processed_count} / {remaining} new rows so far.")

        # Optional small delay between batches
        await asyncio.sleep(DELAY)

    print(f"\n✅ Finished! Processed {processed_count} new rows.")
    print(f"File saved to: {outputFile}")



In [12]:
# %%time
# asyncio.run(generate_cot_with_resume())

## Generate COT Data single call version

In [13]:
def generate_cot_data(prompt: str, answer: str) -> str:
    """Synchronous version (for easier use in loops)"""
    system_prompt = """You are an expert at discovering hidden transformation rules in Alice's Wonderland puzzles.

Think step by step inside <think> </think> tags.
Focus only on explaining how to discover the hidden rule.
Do NOT output the final answer yourself."""

    user_message = f"""Puzzle:
{prompt}

Correct Answer: {answer}

Please think step by step inside <think> tags about how to discover the transformation rule."""

    try:
        response = asyncio.run (acompletion(
            model="openai/local-model",  # Prefix with 'openai/' so LiteLLM uses the OpenAI structure
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.3,
            max_tokens=1600,
            timeout=180
            # reasoning_effort="medium"
        ))
        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # === Hardcode the final answer (Most Reliable) ===
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"

        return final_output
        
    except Exception as e:
        print(f"Error generating CoT for prompt: {e}")
        # Fallback: still return something usable
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"
                            

In [14]:
def generate_cot_data2(prompt: str, answer: str) -> str:
    """Generate high-quality Chain-of-Thought reasoning for puzzle transformation rules."""
    
    system_prompt = """You are an expert puzzle solver specializing in discovering hidden transformation rules in Alice's Wonderland puzzles.

Your task is to carefully analyze the given examples and figure out the secret rule that transforms the input into the output.

Guidelines:
- Think step by step inside <think> </think> tags.
- Focus on identifying the underlying transformation pattern (e.g., bit manipulation, substitution cipher, mathematical formula, string operation, etc.).
- Explain your reasoning clearly: observe the examples, form a hypothesis about the rule, and verify it.
- Do NOT output the final answer yourself. The final answer will be added separately."""

    user_message = f"""Here is a puzzle with several input → output examples. A secret transformation rule is being applied.

{prompt}

The correct output for the last input is: {answer}

Please analyze the examples carefully and think step by step about what the hidden transformation rule might be.

Write your reasoning inside <think> </think> tags. Focus on discovering the pattern."""

    try:
        response = asyncio.run(acompletion(
            model="openai/local-model",
            api_base=LM_STUDIO_BASE_URL,
            api_key=api_key,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message}
            ],
            temperature=0.4,           # Slightly higher for more creative reasoning
            max_tokens=1800,
            timeout=180
        ))

        message = response.choices[0].message
        reasoning = getattr(message, "reasoning_content", "") or ""
        content = message.content or ""

        # Combine reasoning into <think> tags
        if reasoning:
            thinking_part = f"<think>\n{reasoning.strip()}\n</think>"
        else:
            thinking_part = f"<think>\n{content.strip()}\n</think>"

        # Hardcode the final answer (most reliable)
        final_output = f"{thinking_part}\n\\boxed{{{answer}}}"
        return final_output

    except Exception as e:
        print(f"Error generating CoT: {e}")
        return f"<think>\nUnable to generate reasoning.\n</think>\n\\boxed{{{answer}}}"

In [15]:
testFile ="../src/Dataset/test.csv"
trainFile = "../src/Dataset/train.csv"
cotFile = "train_cot.csv"


In [16]:
# trainDF = pd.read_csv(trainFile)
# trainDF

In [17]:
# # Add new column for CoT reasoning
# if "cot_reasoning" not in trainDF.columns:
#     trainDF["cot_reasoning"] = ""

In [18]:
# trainDF

In [19]:
outputFile = "train_cot2.csv"               # output file with CoT
# MODEL = "gpt-4o"                         # or "claude-3-5-sonnet-20241022"
DELAY = 0.1                               # seconds between API calls (adjust based on rate limit)

In [20]:
print("Checking for existing train_cot.csv...")

if os.path.exists(outputFile):
    print("Found existing train_cot.csv → Resuming...")
    trainDF = pd.read_csv(outputFile)
else:
    print("No existing file found. Starting from train.csv...")
    trainDF = pd.read_csv(trainFile)
    if "cot_reasoning" not in trainDF.columns:
        trainDF["cot_reasoning"] = ""

# Count how many rows still need processing
remaining = trainDF["cot_reasoning"].isna().sum() + (trainDF["cot_reasoning"] == "").sum()
print(f"Total rows: {len(trainDF)}")
print(f"Rows already processed: {len(trainDF) - remaining}")
print(f"Rows left to process: {remaining}\n")

Checking for existing train_cot.csv...
Found existing train_cot.csv → Resuming...
Total rows: 9500
Rows already processed: 7145
Rows left to process: 2355



In [21]:
trainDF

,id,prompt,answer,cot_reasoning
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111,<think>\nHere's a thinking process that leads ...
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011,<think>\nThe user wants me to solve a puzzle b...
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book,<think>\nThe user wants me to explain the proc...
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII,<think>\nThe user wants me to identify the hid...
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret,<think>\nHere's a thinking process that leads ...
...,...,...,...,...
9495,ffce9e31,"In Alice's Wonderland, a secret bit manipulati...",01100110,NaN
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",32.45,NaN
9497,ffd89354,"In Alice's Wonderland, secret encryption rules...",student sees the curious mirror,NaN
9498,ffdfb678,"In Alice's Wonderland, secret encryption rules...",the curious mouse creates,NaN


In [22]:
%%time
if remaining == 0:
    print("✅ All rows already have CoT reasoning. Nothing to do.")
else:
    print("Starting CoT generation (resume mode)...\n")

    processed_count = 0

    for idx in tqdm(range(len(trainDF))):
        current_cot = trainDF.loc[idx, "cot_reasoning"]

        # Skip if already has content
        if pd.notna(current_cot) and str(current_cot).strip() != "":
            continue

        prompt = trainDF.loc[idx, "prompt"]
        answer = str(trainDF.loc[idx, "answer"]).strip()

        cot = generate_cot_data2(prompt, answer)
        trainDF.loc[idx, "cot_reasoning"] = cot
        processed_count += 1

        # Save progress every 50 new rows
        if processed_count % 20 == 0:
            trainDF.to_csv(outputFile, index=False)
            print(f"Saved progress. Processed {processed_count} new rows so far.")

        time.sleep(DELAY)

    #concurrent version:
    

    # Final save
    trainDF.to_csv(outputFile, index=False)
    print(f"\n✅ Finished! Processed {processed_count} new rows.")
    print(f"File saved to: {outputFile}")

Starting CoT generation (resume mode)...



 75%|█████████████████████████████████████████████████████████████████▌                     | 7165/9500 [21:53<12:22:37, 19.08s/it]

Saved progress. Processed 20 new rows so far.


 76%|█████████████████████████████████████████████████████████████████▊                     | 7185/9500 [43:39<41:35:40, 64.68s/it]

Saved progress. Processed 40 new rows so far.


...

 79%|██████████████████████████████████████████████████████████████████▉                  | 7485/9500 [5:52:31<36:17:12, 64.83s/it]

Saved progress. Processed 340 new rows so far.


 79%|███████████████████████████████████████████████████████████████████▊                  | 7488/9500 [5:56:13<1:35:42,  2.85s/it]
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x78f5b20be420>


CPU times: user 6.21 s, sys: 654 ms, total: 6.86 s
Wall time: 5h 56min 13s


KeyboardInterrupt: 

In [23]:
# word = "The user wants me to find a transformation rule that maps an 8-bit binary input to an 8-bit binary output based on provided examples.\nThen I need to apply this rule to a new input `00100000` and verify if it matches the expected output `10110111`.\n\nLet's list the examples:\nInput (I) -> Output (O)\n1. 00101011 -> 01100011\n2. 11100000 -> 00000111\n3. 01100001 -> 01100101\n4. 01000111 -> 10100001\n5. 00010010 -> 01011111\n6. 01100010 -> 10100011\n7. 00001100 -> 11100100\n8. 01111010 -> 10010101\n9. 11111010 -> 10110111\n10. 11000111 -> 10000011\n\nTarget Input: 00100000\nExpected Output: 10110111\n\nLet's analyze bit by bit or look for patterns like shifts, XOR with a constant, etc.\n\nExample 1: I = 00101011, O = 01100011\nI bits: b7 b6 b5 b4 b3 b2 b1 b0 (MSB to LSB)\n00101011 -> 0,0,1,0,1,0,1,1\n01100011 -> 0,1,1,0,0,0,1,1\n\nLet's look at differences.\nb7: 0->0\nb6: 0->1 (changed)\nb5: 1->1\nb4: 0->0\nb3: 1->0 (changed)\nb2: 0->0\nb1: 1->1\nb0: 1->1\n\nExample 2: I = 11100000, O = 00000111\nI: 1,1,1,0,0,0,0,0\nO: 0,0,0,0,0,1,1,1\n\nThis looks like a bit rotation or shift.\nInput has three 1s at the start (MSB side). Output has three 1s at the end (LSB side).\nIf we rotate right by 5? Or left by 3?\nLet's check rotation.\nI: 11100000\nRotate Left 3: 00000111. Yes, this matches O.\nLet's test this hypothesis (Rotate Left 3) on other examples.\n\nHypothesis: Output = Input rotated left by $k$ bits? Or maybe right?\nEx 2 suggests Rotate Left 3 or Right 5.\n\nCheck Ex 1 with Rot L 3:\nI: 00101011\nRot L 3: 01011001\nO is 01100011. No match.\nSo it's not a simple global rotation for all inputs.\n\nLet's look at bit positions. Maybe each output bit depends on specific input bits (linear combination or permutation).\nOr maybe there's an arithmetic operation?\nI: 43, O: 99. Diff = +56.\nI: 224, O: 7. Diff = -217 (or +239 mod 256).\nI: 97, O: 101. Diff = +4.\nNo obvious arithmetic pattern like $x+c$ or $x \\times k$.\n\nLet's look at bitwise operations.\nMaybe XOR with a constant?\nEx 1: I=00101011, O=01100011.\nXOR = 01001000 (72).\nEx 2: I=11100000, O=00000111.\nXOR = 11100111 (231).\nConstants are different.\n\nMaybe XOR with a shifted version of itself? Or some function of bits.\nLet's look at the relationship between specific bit positions.\nLet $I_i$ be input bit at position $i$ (7 down to 0).\nLet $O_i$ be output bit at position $i$.\n\nEx 1: I=00101011, O=01100011\nI bits: 0 0 1 0 1 0 1 1\nO bits: 0 1 1 0 0 0 1 1\n\nLet's look at Ex 2 again. I=11100000, O=00000111.\nI bits: 1 1 1 0 0 0 0 0\nO bits: 0 0 0 0 0 1 1 1\n\nIt looks like the block of 1s moved from left to right.\nIn Ex 2, input has 1s at indices 7,6,5. Output has 1s at indices 2,1,0.\nShift amount seems related to value? Or fixed shift for some bits?\n\nLet's look at Ex 4: I=01000111 (71), O=10100001 (161).\nI: 0 1 0 0 0 1 1 1\nO: 1 0 1 0 0 0 0 1\n\nLet's look at Ex 8: I=01111010, O=10010101.\nI: 0 1 1 1 1 0 1 0\nO: 1 0 0 1 0 1 0 1\n\nLet's look at Ex 9: I=11111010, O=10110111. (This is the target output for our query input!)\nI: 1 1 1 1 1 0 1 0\nO: 1 0 1 1 0 1 1 1\n\nLet's compare Ex 9 and Target Input/Output.\nTarget I: 00100000 -> O: 10110111 (Given in prompt).\nWait, the prompt says \"The correct output for the last input is: 10110111\".\nThis means $f(00100000) = 10110111$.\n\nLet's look at Ex 9 again.\nI: 11111010 -> O: 10110111.\nTarget I: 00100000 -> O: 10110111.\nWait, if $f(11111010) = 10110111$ and $f(00100000) = 10110111$, then the function is not injective? Or maybe I misread the target.\nThe prompt says: \"Now, determine the output for: 00100000 ... The correct output for the last input is: 10110111\".\nThis"

In [24]:
# Concurrent

In [25]:
# len(word)

In [26]:
trainDF.to_csv(outputFile, index=False)
# print(f"\n✅ Finished! Processed {processed_count} new rows.")
# print(f"File saved to: {outputFile}")